In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))


In [3]:
# GPU 활성화 끄기
# import os
# os.environ['CUDA_VISIBLE_DEVICES'] ="-1"
# GPU 사용 여부 확인
import tensorflow as tf
print(tf.__version__)
tf.config.list_physical_devices('GPU')


2.10.0


[]

# 1. 기존의 프로그램 방식 

In [4]:
import numpy as np
import matplotlib.pyplot as plt


In [5]:
# 섭씨온도(input data)를 받아 화씨 온도로 출력
def celsius_to_faherenheit(c):
    return 1.8*c + 32

In [11]:
input_data = int(input('섭씨 온도는? '))
print('화씨 온도는 ', celsius_to_faherenheit(input_data))

섭씨 온도는? 1
화씨 온도는  33.8


# 2.머신러닝 / 딥러닝 프로그램 방식
- 1. 데이터 확보 및 생성
- 2. 데이터 전처리 : scale조정, 라벨링처리, 훈련데이터셋(학습데이터셋), 검증데이터셋, 시럼데이터셋으로 나누기
- 3. 모델구성
- 4. 모델 학습과정 설정
- 5. 모델 학습스키기 (학습데이터셋과 검증데이터셋)
- 6. 모델 평가(시험데이터셋)
- 7. 모델 사용(입력값이 주어지면 예측값을 받기)
## 2.1 노이즈가 없는 데이터로 실습

In [12]:
# 1. 데이터 생성
data_C = np.arange(100)
data_C # 입력변수==독립변수

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84,
       85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99])

In [13]:
data_F = celsius_to_faherenheit(data_C)
data_F

array([ 32. ,  33.8,  35.6,  37.4,  39.2,  41. ,  42.8,  44.6,  46.4,
        48.2,  50. ,  51.8,  53.6,  55.4,  57.2,  59. ,  60.8,  62.6,
        64.4,  66.2,  68. ,  69.8,  71.6,  73.4,  75.2,  77. ,  78.8,
        80.6,  82.4,  84.2,  86. ,  87.8,  89.6,  91.4,  93.2,  95. ,
        96.8,  98.6, 100.4, 102.2, 104. , 105.8, 107.6, 109.4, 111.2,
       113. , 114.8, 116.6, 118.4, 120.2, 122. , 123.8, 125.6, 127.4,
       129.2, 131. , 132.8, 134.6, 136.4, 138.2, 140. , 141.8, 143.6,
       145.4, 147.2, 149. , 150.8, 152.6, 154.4, 156.2, 158. , 159.8,
       161.6, 163.4, 165.2, 167. , 168.8, 170.6, 172.4, 174.2, 176. ,
       177.8, 179.6, 181.4, 183.2, 185. , 186.8, 188.6, 190.4, 192.2,
       194. , 195.8, 197.6, 199.4, 201.2, 203. , 204.8, 206.6, 208.4,
       210.2])

In [15]:
# 2. 데이터 전처리 : 컴퓨터에게 학습시키기 위해, Normallize함(변수들의 편차를 비슷하게)
scaled_data_C = data_C/100.0
scaled_data_F = data_F/100.0
print('독립변수는 ', scaled_data_C)
print('종속변수는 ', scaled_data_F)

독립변수는  [0.   0.01 0.02 0.03 0.04 0.05 0.06 0.07 0.08 0.09 0.1  0.11 0.12 0.13
 0.14 0.15 0.16 0.17 0.18 0.19 0.2  0.21 0.22 0.23 0.24 0.25 0.26 0.27
 0.28 0.29 0.3  0.31 0.32 0.33 0.34 0.35 0.36 0.37 0.38 0.39 0.4  0.41
 0.42 0.43 0.44 0.45 0.46 0.47 0.48 0.49 0.5  0.51 0.52 0.53 0.54 0.55
 0.56 0.57 0.58 0.59 0.6  0.61 0.62 0.63 0.64 0.65 0.66 0.67 0.68 0.69
 0.7  0.71 0.72 0.73 0.74 0.75 0.76 0.77 0.78 0.79 0.8  0.81 0.82 0.83
 0.84 0.85 0.86 0.87 0.88 0.89 0.9  0.91 0.92 0.93 0.94 0.95 0.96 0.97
 0.98 0.99]
종속변수는  [0.32  0.338 0.356 0.374 0.392 0.41  0.428 0.446 0.464 0.482 0.5   0.518
 0.536 0.554 0.572 0.59  0.608 0.626 0.644 0.662 0.68  0.698 0.716 0.734
 0.752 0.77  0.788 0.806 0.824 0.842 0.86  0.878 0.896 0.914 0.932 0.95
 0.968 0.986 1.004 1.022 1.04  1.058 1.076 1.094 1.112 1.13  1.148 1.166
 1.184 1.202 1.22  1.238 1.256 1.274 1.292 1.31  1.328 1.346 1.364 1.382
 1.4   1.418 1.436 1.454 1.472 1.49  1.508 1.526 1.544 1.562 1.58  1.598
 1.616 1.634 1.652 1.67  1.688 1.706 1.7

In [16]:
# 3. 모델 구성 (tensorflow)
from tensorflow.keras.models import Sequential # 모델 생성
from tensorflow.keras.layers import Dense, Input # 입력값과 출력값으로 layer층 지정
model =Sequential()
model.add(Dense(1, # 출력(종속, 타겟)변수의 갯수 
                input_shape=(1,)
                ))
print(model.summary())

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 1)                 2         
                                                                 
Total params: 2
Trainable params: 2
Non-trainable params: 0
_________________________________________________________________
None


- 회귀분석 오치함수 :
    MSE(오차제곱평균), RMSE(루트를 취하기 때문에 MSE단점이 어느정도 해소. 덜 민감), MAE(절대값평균)

In [21]:
# 4. 모델 학습 과정 설정
model.compile(loss='mse', optimizer='rmsprop', metrics=['mae'])
              # 손실함수   옵티마이저             평가지표
# loss(오차, 손실함수)는 모델 학습 중 최적화할 대상
# metrics : 평가지표

In [22]:
# 학습 전 예측 
model.predict(np.array([[0],
                       [0.01]]))

1/1 [==============================] - 0s 37ms/step


array([[ 0.       ],
       [-0.0136154]], dtype=float32)

In [23]:
# 학습 전 모델 저장
model.save('model/before_learning.h5') # kermel에 w(기울기), bias에 b이 있음
# from tnesorflow.keras.model import save_model
# save_model(model, 'model/before_learning.h5')

In [24]:
# 모델 학습 시키기 - 1번만 실행
hist = model.fit(scaled_data_C, scaled_data_F, epochs=1000, verbose=2)
                 # 독립변수       타겟변수(종속) 학습횟수    학습시출력여부

Epoch 1/1000
4/4 - 0s - loss: 4.3693 - mae: 1.8806 - 409ms/epoch - 102ms/step
Epoch 2/1000
4/4 - 0s - loss: 4.3248 - mae: 1.8700 - 10ms/epoch - 2ms/step
Epoch 3/1000
4/4 - 0s - loss: 4.2927 - mae: 1.8621 - 7ms/epoch - 2ms/step
Epoch 4/1000
4/4 - 0s - loss: 4.2653 - mae: 1.8553 - 8ms/epoch - 2ms/step
Epoch 5/1000
4/4 - 0s - loss: 4.2369 - mae: 1.8484 - 7ms/epoch - 2ms/step
Epoch 6/1000
4/4 - 0s - loss: 4.2107 - mae: 1.8420 - 7ms/epoch - 2ms/step
Epoch 7/1000
4/4 - 0s - loss: 4.1838 - mae: 1.8351 - 7ms/epoch - 2ms/step
Epoch 8/1000
4/4 - 0s - loss: 4.1597 - mae: 1.8293 - 6ms/epoch - 2ms/step
Epoch 9/1000
4/4 - 0s - loss: 4.1359 - mae: 1.8233 - 8ms/epoch - 2ms/step
Epoch 10/1000
4/4 - 0s - loss: 4.1115 - mae: 1.8172 - 8ms/epoch - 2ms/step
Epoch 11/1000
4/4 - 0s - loss: 4.0868 - mae: 1.8110 - 7ms/epoch - 2ms/step
Epoch 12/1000
4/4 - 0s - loss: 4.0625 - mae: 1.8049 - 6ms/epoch - 1ms/step
Epoch 13/1000
4/4 - 0s - loss: 4.0400 - mae: 1.7990 - 6ms/epoch - 1ms/step
Epoch 14/1000
4/4 - 0s - loss

Epoch 110/1000
4/4 - 0s - loss: 2.1481 - mae: 1.2416 - 9ms/epoch - 2ms/step
Epoch 111/1000
4/4 - 0s - loss: 2.1317 - mae: 1.2361 - 11ms/epoch - 3ms/step
Epoch 112/1000
4/4 - 0s - loss: 2.1135 - mae: 1.2299 - 10ms/epoch - 2ms/step
Epoch 113/1000
4/4 - 0s - loss: 2.0983 - mae: 1.2248 - 8ms/epoch - 2ms/step
Epoch 114/1000
4/4 - 0s - loss: 2.0822 - mae: 1.2193 - 9ms/epoch - 2ms/step
Epoch 115/1000
4/4 - 0s - loss: 2.0664 - mae: 1.2139 - 9ms/epoch - 2ms/step
Epoch 116/1000
4/4 - 0s - loss: 2.0530 - mae: 1.2094 - 10ms/epoch - 3ms/step
Epoch 117/1000
4/4 - 0s - loss: 2.0371 - mae: 1.2038 - 10ms/epoch - 2ms/step
Epoch 118/1000
4/4 - 0s - loss: 2.0214 - mae: 1.1986 - 9ms/epoch - 2ms/step
Epoch 119/1000
4/4 - 0s - loss: 2.0037 - mae: 1.1924 - 8ms/epoch - 2ms/step
Epoch 120/1000
4/4 - 0s - loss: 1.9895 - mae: 1.1875 - 8ms/epoch - 2ms/step
Epoch 121/1000
4/4 - 0s - loss: 1.9745 - mae: 1.1823 - 8ms/epoch - 2ms/step
Epoch 122/1000
4/4 - 0s - loss: 1.9588 - mae: 1.1768 - 9ms/epoch - 2ms/step
Epoch 12

Epoch 218/1000
4/4 - 0s - loss: 0.8262 - mae: 0.7428 - 7ms/epoch - 2ms/step
Epoch 219/1000
4/4 - 0s - loss: 0.8182 - mae: 0.7392 - 9ms/epoch - 2ms/step
Epoch 220/1000
4/4 - 0s - loss: 0.8101 - mae: 0.7357 - 10ms/epoch - 2ms/step
Epoch 221/1000
4/4 - 0s - loss: 0.8013 - mae: 0.7318 - 8ms/epoch - 2ms/step
Epoch 222/1000
4/4 - 0s - loss: 0.7949 - mae: 0.7290 - 8ms/epoch - 2ms/step
Epoch 223/1000
4/4 - 0s - loss: 0.7864 - mae: 0.7252 - 7ms/epoch - 2ms/step
Epoch 224/1000
4/4 - 0s - loss: 0.7790 - mae: 0.7220 - 9ms/epoch - 2ms/step
Epoch 225/1000
4/4 - 0s - loss: 0.7708 - mae: 0.7183 - 10ms/epoch - 2ms/step
Epoch 226/1000
4/4 - 0s - loss: 0.7641 - mae: 0.7153 - 9ms/epoch - 2ms/step
Epoch 227/1000
4/4 - 0s - loss: 0.7566 - mae: 0.7119 - 8ms/epoch - 2ms/step
Epoch 228/1000
4/4 - 0s - loss: 0.7490 - mae: 0.7086 - 9ms/epoch - 2ms/step
Epoch 229/1000
4/4 - 0s - loss: 0.7421 - mae: 0.7054 - 9ms/epoch - 2ms/step
Epoch 230/1000
4/4 - 0s - loss: 0.7351 - mae: 0.7025 - 10ms/epoch - 2ms/step
Epoch 231

Epoch 326/1000
4/4 - 0s - loss: 0.3143 - mae: 0.4840 - 9ms/epoch - 2ms/step
Epoch 327/1000
4/4 - 0s - loss: 0.3129 - mae: 0.4830 - 7ms/epoch - 2ms/step
Epoch 328/1000
4/4 - 0s - loss: 0.3109 - mae: 0.4816 - 8ms/epoch - 2ms/step
Epoch 329/1000
4/4 - 0s - loss: 0.3096 - mae: 0.4808 - 8ms/epoch - 2ms/step
Epoch 330/1000
4/4 - 0s - loss: 0.3082 - mae: 0.4798 - 10ms/epoch - 3ms/step
Epoch 331/1000
4/4 - 0s - loss: 0.3071 - mae: 0.4789 - 7ms/epoch - 2ms/step
Epoch 332/1000
4/4 - 0s - loss: 0.3058 - mae: 0.4781 - 8ms/epoch - 2ms/step
Epoch 333/1000
4/4 - 0s - loss: 0.3038 - mae: 0.4766 - 9ms/epoch - 2ms/step
Epoch 334/1000
4/4 - 0s - loss: 0.3019 - mae: 0.4751 - 9ms/epoch - 2ms/step
Epoch 335/1000
4/4 - 0s - loss: 0.3004 - mae: 0.4741 - 10ms/epoch - 2ms/step
Epoch 336/1000
4/4 - 0s - loss: 0.2994 - mae: 0.4733 - 6ms/epoch - 1ms/step
Epoch 337/1000
4/4 - 0s - loss: 0.2981 - mae: 0.4723 - 6ms/epoch - 2ms/step
Epoch 338/1000
4/4 - 0s - loss: 0.2971 - mae: 0.4716 - 8ms/epoch - 2ms/step
Epoch 339/

Epoch 434/1000
4/4 - 0s - loss: 0.2023 - mae: 0.3887 - 8ms/epoch - 2ms/step
Epoch 435/1000
4/4 - 0s - loss: 0.2015 - mae: 0.3879 - 7ms/epoch - 2ms/step
Epoch 436/1000
4/4 - 0s - loss: 0.2005 - mae: 0.3870 - 12ms/epoch - 3ms/step
Epoch 437/1000
4/4 - 0s - loss: 0.1996 - mae: 0.3860 - 9ms/epoch - 2ms/step
Epoch 438/1000
4/4 - 0s - loss: 0.1990 - mae: 0.3853 - 7ms/epoch - 2ms/step
Epoch 439/1000
4/4 - 0s - loss: 0.1982 - mae: 0.3846 - 9ms/epoch - 2ms/step
Epoch 440/1000
4/4 - 0s - loss: 0.1974 - mae: 0.3838 - 8ms/epoch - 2ms/step
Epoch 441/1000
4/4 - 0s - loss: 0.1966 - mae: 0.3830 - 10ms/epoch - 3ms/step
Epoch 442/1000
4/4 - 0s - loss: 0.1958 - mae: 0.3822 - 10ms/epoch - 3ms/step
Epoch 443/1000
4/4 - 0s - loss: 0.1951 - mae: 0.3814 - 8ms/epoch - 2ms/step
Epoch 444/1000
4/4 - 0s - loss: 0.1943 - mae: 0.3807 - 7ms/epoch - 2ms/step
Epoch 445/1000
4/4 - 0s - loss: 0.1935 - mae: 0.3800 - 6ms/epoch - 1ms/step
Epoch 446/1000
4/4 - 0s - loss: 0.1928 - mae: 0.3793 - 8ms/epoch - 2ms/step
Epoch 447

Epoch 542/1000
4/4 - 0s - loss: 0.1194 - mae: 0.2984 - 7ms/epoch - 2ms/step
Epoch 543/1000
4/4 - 0s - loss: 0.1188 - mae: 0.2976 - 8ms/epoch - 2ms/step
Epoch 544/1000
4/4 - 0s - loss: 0.1180 - mae: 0.2967 - 8ms/epoch - 2ms/step
Epoch 545/1000
4/4 - 0s - loss: 0.1174 - mae: 0.2959 - 8ms/epoch - 2ms/step
Epoch 546/1000
4/4 - 0s - loss: 0.1167 - mae: 0.2949 - 6ms/epoch - 2ms/step
Epoch 547/1000
4/4 - 0s - loss: 0.1160 - mae: 0.2942 - 7ms/epoch - 2ms/step
Epoch 548/1000
4/4 - 0s - loss: 0.1154 - mae: 0.2934 - 6ms/epoch - 2ms/step
Epoch 549/1000
4/4 - 0s - loss: 0.1148 - mae: 0.2925 - 7ms/epoch - 2ms/step
Epoch 550/1000
4/4 - 0s - loss: 0.1141 - mae: 0.2917 - 6ms/epoch - 1ms/step
Epoch 551/1000
4/4 - 0s - loss: 0.1134 - mae: 0.2908 - 5ms/epoch - 1ms/step
Epoch 552/1000
4/4 - 0s - loss: 0.1128 - mae: 0.2901 - 7ms/epoch - 2ms/step
Epoch 553/1000
4/4 - 0s - loss: 0.1121 - mae: 0.2891 - 6ms/epoch - 2ms/step
Epoch 554/1000
4/4 - 0s - loss: 0.1115 - mae: 0.2882 - 10ms/epoch - 2ms/step
Epoch 555/1

Epoch 650/1000
4/4 - 0s - loss: 0.0578 - mae: 0.2075 - 7ms/epoch - 2ms/step
Epoch 651/1000
4/4 - 0s - loss: 0.0574 - mae: 0.2068 - 5ms/epoch - 1ms/step
Epoch 652/1000
4/4 - 0s - loss: 0.0568 - mae: 0.2057 - 8ms/epoch - 2ms/step
Epoch 653/1000
4/4 - 0s - loss: 0.0563 - mae: 0.2048 - 7ms/epoch - 2ms/step
Epoch 654/1000
4/4 - 0s - loss: 0.0558 - mae: 0.2039 - 5ms/epoch - 1ms/step
Epoch 655/1000
4/4 - 0s - loss: 0.0554 - mae: 0.2030 - 7ms/epoch - 2ms/step
Epoch 656/1000
4/4 - 0s - loss: 0.0550 - mae: 0.2023 - 8ms/epoch - 2ms/step
Epoch 657/1000
4/4 - 0s - loss: 0.0546 - mae: 0.2015 - 7ms/epoch - 2ms/step
Epoch 658/1000
4/4 - 0s - loss: 0.0542 - mae: 0.2008 - 6ms/epoch - 1ms/step
Epoch 659/1000
4/4 - 0s - loss: 0.0538 - mae: 0.2001 - 8ms/epoch - 2ms/step
Epoch 660/1000
4/4 - 0s - loss: 0.0533 - mae: 0.1992 - 6ms/epoch - 1ms/step
Epoch 661/1000
4/4 - 0s - loss: 0.0528 - mae: 0.1982 - 9ms/epoch - 2ms/step
Epoch 662/1000
4/4 - 0s - loss: 0.0524 - mae: 0.1975 - 9ms/epoch - 2ms/step
Epoch 663/10

Epoch 758/1000
4/4 - 0s - loss: 0.0188 - mae: 0.1182 - 10ms/epoch - 2ms/step
Epoch 759/1000
4/4 - 0s - loss: 0.0185 - mae: 0.1174 - 10ms/epoch - 2ms/step
Epoch 760/1000
4/4 - 0s - loss: 0.0182 - mae: 0.1164 - 11ms/epoch - 3ms/step
Epoch 761/1000
4/4 - 0s - loss: 0.0179 - mae: 0.1154 - 11ms/epoch - 3ms/step
Epoch 762/1000
4/4 - 0s - loss: 0.0176 - mae: 0.1146 - 13ms/epoch - 3ms/step
Epoch 763/1000
4/4 - 0s - loss: 0.0173 - mae: 0.1137 - 9ms/epoch - 2ms/step
Epoch 764/1000
4/4 - 0s - loss: 0.0171 - mae: 0.1127 - 11ms/epoch - 3ms/step
Epoch 765/1000
4/4 - 0s - loss: 0.0167 - mae: 0.1117 - 12ms/epoch - 3ms/step
Epoch 766/1000
4/4 - 0s - loss: 0.0165 - mae: 0.1108 - 11ms/epoch - 3ms/step
Epoch 767/1000
4/4 - 0s - loss: 0.0163 - mae: 0.1101 - 11ms/epoch - 3ms/step
Epoch 768/1000
4/4 - 0s - loss: 0.0161 - mae: 0.1095 - 11ms/epoch - 3ms/step
Epoch 769/1000
4/4 - 0s - loss: 0.0159 - mae: 0.1089 - 7ms/epoch - 2ms/step
Epoch 770/1000
4/4 - 0s - loss: 0.0157 - mae: 0.1080 - 10ms/epoch - 2ms/step
E

Epoch 866/1000
4/4 - 0s - loss: 0.0012 - mae: 0.0305 - 18ms/epoch - 4ms/step
Epoch 867/1000
4/4 - 0s - loss: 0.0012 - mae: 0.0296 - 7ms/epoch - 2ms/step
Epoch 868/1000
4/4 - 0s - loss: 0.0011 - mae: 0.0287 - 8ms/epoch - 2ms/step
Epoch 869/1000
4/4 - 0s - loss: 0.0010 - mae: 0.0278 - 9ms/epoch - 2ms/step
Epoch 870/1000
4/4 - 0s - loss: 9.9352e-04 - mae: 0.0272 - 7ms/epoch - 2ms/step
Epoch 871/1000
4/4 - 0s - loss: 9.3960e-04 - mae: 0.0265 - 8ms/epoch - 2ms/step
Epoch 872/1000
4/4 - 0s - loss: 8.9008e-04 - mae: 0.0257 - 8ms/epoch - 2ms/step
Epoch 873/1000
4/4 - 0s - loss: 8.4736e-04 - mae: 0.0250 - 7ms/epoch - 2ms/step
Epoch 874/1000
4/4 - 0s - loss: 8.0181e-04 - mae: 0.0245 - 7ms/epoch - 2ms/step
Epoch 875/1000
4/4 - 0s - loss: 7.6135e-04 - mae: 0.0238 - 5ms/epoch - 1ms/step
Epoch 876/1000
4/4 - 0s - loss: 7.2666e-04 - mae: 0.0233 - 11ms/epoch - 3ms/step
Epoch 877/1000
4/4 - 0s - loss: 6.8182e-04 - mae: 0.0226 - 6ms/epoch - 1ms/step
Epoch 878/1000
4/4 - 0s - loss: 6.3094e-04 - mae: 0.02

Epoch 966/1000
4/4 - 0s - loss: 1.6050e-07 - mae: 3.9260e-04 - 8ms/epoch - 2ms/step
Epoch 967/1000
4/4 - 0s - loss: 6.2269e-07 - mae: 7.6891e-04 - 5ms/epoch - 1ms/step
Epoch 968/1000
4/4 - 0s - loss: 1.0892e-06 - mae: 0.0010 - 8ms/epoch - 2ms/step
Epoch 969/1000
4/4 - 0s - loss: 1.4563e-07 - mae: 3.7328e-04 - 9ms/epoch - 2ms/step
Epoch 970/1000
4/4 - 0s - loss: 5.0516e-07 - mae: 6.9205e-04 - 9ms/epoch - 2ms/step
Epoch 971/1000
4/4 - 0s - loss: 1.2384e-06 - mae: 0.0011 - 9ms/epoch - 2ms/step
Epoch 972/1000
4/4 - 0s - loss: 3.7804e-07 - mae: 5.8529e-04 - 7ms/epoch - 2ms/step
Epoch 973/1000
4/4 - 0s - loss: 4.3627e-07 - mae: 6.3294e-04 - 10ms/epoch - 2ms/step
Epoch 974/1000
4/4 - 0s - loss: 1.8305e-07 - mae: 4.0848e-04 - 11ms/epoch - 3ms/step
Epoch 975/1000
4/4 - 0s - loss: 7.4349e-07 - mae: 8.2996e-04 - 13ms/epoch - 3ms/step
Epoch 976/1000
4/4 - 0s - loss: 1.2916e-06 - mae: 0.0010 - 11ms/epoch - 3ms/step
Epoch 977/1000
4/4 - 0s - loss: 4.6438e-08 - mae: 2.1012e-04 - 9ms/epoch - 2ms/step
